# Real-Time Audio Deepfake Interception System - Training Notebook
This notebook runs dataset preparation, downsampling (16kHz to 8kHz), G.711 phone codec & noise augmentation, PyTorch model training, evaluation (Accuracy & EER), and ONNX model export using Google Colab's GPU.

In [ ]:
# Step 1: Verify GPU Acceleration in Colab
!nvidia-smi
import torch
print("PyTorch GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# Step 2: Install required libraries
!pip install -q librosa scikit-learn onnx onnxruntime matplotlib

In [ ]:
# Step 3: Mount Google Drive (Optional - if your datasets are in Google Drive)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 4: Define Automatic 16kHz -> 8kHz On-the-Fly Audio Downsampler & Feature Extractor
import os
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import audioop
import random

SAMPLE_RATE = 8000 # Downsampling target
SAMPLES = 8000      # 1 second clip at 8kHz

def load_and_resample(audio_path):
    '''Reads 16kHz studio audio and resamples to 8kHz in memory automatically'''
    try:
        audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
        if len(audio) < SAMPLES:
            audio = np.pad(audio, (0, SAMPLES - len(audio)))
        else:
            audio = audio[:SAMPLES]
        return audio.astype(np.float32)
    except Exception as e:
        return None

def simulate_g711_phone(audio):
    '''Simulates G.711 mu-law telephone codec compression'''
    audio = np.clip(audio, -1.0, 1.0)
    as_int16 = (audio * 32767).astype(np.int16).tobytes()
    encoded = audioop.lin2ulaw(as_int16, 2)
    decoded = audioop.ulaw2lin(encoded, 2)
    result = np.frombuffer(decoded, dtype=np.int16).astype(np.float32)
    return result / 32767.0

In [ ]:
# Step 5: Lightweight Deepfake Classifier Model (LCNN Architecture)
class AudioDeepfakeClassifier(nn.Module):
    def __init__(self):
        super(AudioDeepfakeClassifier, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=15, stride=2, padding=7)
        self.bn1 = nn.BatchNorm1d(16)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)
        
        self.conv2 = nn.Conv1d(16, 32, kernel_size=7, stride=2, padding=3)
        self.bn2 = nn.BatchNorm1d(32)
        
        self.conv3 = nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2)
        self.bn3 = nn.BatchNorm1d(64)
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.global_pool(x).squeeze(-1)
        out = self.fc(x)
        return out

print("Model architecture initialized successfully.")